# RAG-Based Profile Matching Evaluation

This notebook walks through the experimentation, indexing, matching, and performance analysis of the profile matcher.

In [1]:
import sys
import time
import json
from pathlib import Path

# Add src/ to path
sys.path.append(str(Path("..") / "src"))

import config
from resume_rag import ResumeRAGPipeline
from job_matcher import JobMatcher
from fs_tools import list_files, read_file

## Step 1: Ingest the Resumes

First, we initialize the RAG Pipeline and ingest all generated candidate resumes from the data directory.

In [2]:
print("Initializing pipeline...")
pipeline = ResumeRAGPipeline()

print("Ingesting directory...")
start_time = time.time()
pipeline.ingest_directory(config.RESUMES_DIR)
end_time = time.time()

print(f"Ingested all resumes in {end_time - start_time:.2f} seconds.")

Initializing pipeline...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ingesting directory...
Ingesting resume_sarah_connor.txt - Name: Sarah Connor, Exp: 7 yrs, Skills: 8
Ingesting resume_kyle_reese.docx - Name: Kyle Reese, Exp: 4 yrs, Skills: 8
Ingesting resume_bucky_barnes.docx - Name: Bucky Barnes, Exp: 9 yrs, Skills: 6
Ingesting resume_bruce_banner.pdf - Name: Bruce Banner, Exp: 10 yrs, Skills: 7
Ingesting resume_natasha_romanoff.docx - Name: Natasha Romanoff, Exp: 6 yrs, Skills: 6
Ingesting resume_wanda_maximoff.docx - Name: Wanda Maximoff, Exp: 5 yrs, Skills: 8
Ingesting resume_tony_stark.pdf - Name: Tony Stark, Exp: 15 yrs, Skills: 8
Ingesting resume_vision.pdf - Name: Vision, Exp: 8 yrs, Skills: 5
Ingesting resume_steve_rogers.txt - Name: Steve Rogers, Exp: 8 yrs, Skills: 4
Ingesting resume_ellen_ripley.txt - Name: Ellen Ripley, Exp: 12 yrs, Skills: 7
Ingesting resume_diana_prince.txt - Name: Diana Prince, Exp: 8 yrs, Skills: 4
Ingesting resume_clark_kent.pdf - Name: Clark Kent, Exp: 5 yrs, Skills: 8
Ingesting resume_clint_barton.txt - Name: Clin

## Step 2: Query the Matcher with Job Descriptions

Let's match each of the 5 job descriptions against our candidate database. We will record matching latency and display the top 3 candidates for each role.

In [3]:
matcher = JobMatcher()
jds = list_files(config.JOB_DESCRIPTIONS_DIR)

latencies = []

for jd_file in sorted(jds, key=lambda f: f['name']):
    jd_data = read_file(jd_file['path'])
    if not jd_data['success']:
        continue
    
    content = jd_data['content']
    print(f"\n{'='*60}")
    print(f"JD File: {jd_file['name']}")
    print(f"{'='*60}")
    print(content[:200] + "...\n")
    
    # Run Matcher and measure latency
    t0 = time.time()
    results = matcher.match(content, k=3)
    t1 = time.time()
    
    latency = (t1 - t0) * 1000  # in ms
    latencies.append(latency)
    
    print(f"Retrieval latency: {latency:.2f} ms")
    print("Top Matches:")
    for idx, match in enumerate(results['top_matches']):
        print(f"  {idx+1}. {match['candidate_name']} (Score: {match['match_score']})")
        print(f"     Reasoning: {match['reasoning']}")
        print(f"     Excerpts: {match['relevant_excerpts'][0][:150]}...")
        print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


JD File: jd_devops_cloud.txt
Job Description: DevOps & Cloud Infrastructure Architect

We are looking for a DevOps Architect to automate deployments and manage cloud environments.
Requirements:
- Must have 5+ years of experience
...

Auto-detected experience requirement from JD: 5+ years
No explicit must-have skills filter applied.
Retrieval latency: 44.71 ms
Top Matches:
  1. Michael Lee (Score: 85)
     Reasoning: Strong skill overlap for AWS, Docker, Jenkins, Kubernetes, Terraform. Candidate possesses 6 years of experience (education: 'B.S. in Computer Engineering, Berkeley'). Highest matching content was found in sections: EXPERIENCE, SUMMARY.
     Excerpts: Cloud Infrastructure Engineer at ScaleOps (2020-Present)
- Managed AWS infrastructure using Terraform (Infrastructure as Code).
- Implemented robust C...

  2. David Bowman (Score: 78)
     Reasoning: Strong skill overlap for AWS, Docker, Kubernetes, Terraform. Candidate possesses 9 years of experience (education: 'M.S. in Dis

## Step 3: Performance Metrics Analysis

We'll print statistics on retrieval speed.

In [4]:
import numpy as np

print("Matching Performance Metrics:")
print(f"- Average Match Latency: {np.mean(latencies):.2f} ms")
print(f"- Median Match Latency: {np.median(latencies):.2f} ms")
print(f"- Min Match Latency: {np.min(latencies):.2f} ms")
print(f"- Max Match Latency: {np.max(latencies):.2f} ms")

Matching Performance Metrics:
- Average Match Latency: 34.32 ms
- Median Match Latency: 32.67 ms
- Min Match Latency: 28.17 ms
- Max Match Latency: 44.71 ms
